In [1]:
from script.model.pepbridge import PepBridge
from script.model.lora import build_model_with_lora
from script.dataset import PTDataSet, MPDataSet, MPTDataSet, \
    build_loader_for_long_tail, build_loader_uniform_by_peptide
from script.dataprocess import mk_aa_dict, mk_bv_dict
from script.utils import model_fn, encoder_load_state_dict, df_train_test_split, setup_logger
from script.train import train_three_phases_multi_loaders

import pandas as pd 
import numpy as np
import os

import torch
from torch.utils.data import DataLoader
import pickle



In [2]:
with open('./doc/esm_emb_HLAI.pkl', 'rb') as f:
    mhc_dict = pickle.load(f)

In [9]:
mp_test = pd.read_csv('../pMHC/External_mp_test.csv',index_col=0,header=0)

In [18]:
np.asarray(mp_test['len'])

array([ 8,  8,  8, ..., 14, 14, 14])

0          8
1          8
2          8
3          8
4          8
          ..
103860    14
103861    14
103862    14
103863    14
103864    14
Name: len, Length: 102459, dtype: int64

In [11]:
mp_train = pd.read_csv('../pMHC/mp_train.csv',index_col=0,header=0)
mp_train

,MHC,peptide,binding,len
0,HLA-C14:02,YFPLAPFNQL,1,10
1,HLA-B44:02,KESKINQVF,1,9
2,HLA-B54:01,QPHDPLVPLSA,1,11
3,HLA-B57:03,RTIADSLINSF,1,11
4,HLA-C08:02,LLDELPQSVL,1,10
...,...,...,...,...
1487169,HLA-C17:01,QITQRKWE,0,8
1487170,HLA-C17:01,FTAVVLVSAD,0,10
1487171,HLA-C17:01,GWEVKSLR,0,8
1487172,HLA-C17:01,TADSIGRVKA,0,10


In [14]:
row = mp_test.iloc[1]

In [16]:
row['MHC']

'HLA-A01:01'

In [13]:
mp_train.columns

Index(['MHC', 'peptide', 'binding', 'len'], dtype='object')

In [2]:
aa_dict = mk_aa_dict()
bv_dcit = mk_bv_dict()

device = 'mps'

model = model_fn(aa_vocab_size=len(aa_dict),
                trbv_vocab_size=len(bv_dcit))
model.to(device)
model = encoder_load_state_dict(model, peptide_pt_path='./doc/peptide_mlm.pt',
                                cdr3_pt_path='./doc/cdr3_mlm.pt', device=device)

In [3]:
pt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/pt_train.csv', 
                    header=0, index_col=0)
pt_train, pt_val = df_train_test_split(pt_df, val_split=0.2, seed=42)
del pt_df

mp_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/mp_train.csv', 
                    header=0, index_col=0)
mp_train, mp_val = df_train_test_split(mp_df, val_split=0.2, seed=42)
del mp_df

mpt_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC_TCR/dataset/mpt_train.csv', 
                    header=0, index_col=0)
mpt_train, mpt_val = df_train_test_split(mpt_df, val_split=0.2, seed=42)
del mpt_df

align_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/pMHC/random_pMHC_cdr3_train.csv', 
                    header=0, index_col=0,nrows=100000)
align_train, align_val = df_train_test_split(align_df, val_split=0.2, seed=42)
del align_df

In [4]:
imm_df = pd.read_csv('/Users/aapupu/Desktop/vscode/TCR/immuno-genicity/immunogenicity_train.csv', 
                    header=0, index_col=0)
imm_train, imm_val = df_train_test_split(imm_df, val_split=0.2, seed=42)

mp_contact_df = pd.read_csv('./data/mp_pdb_train.csv',  header=0, index_col=0)
mp_contact_train, mp_contact_val = df_train_test_split(mp_contact_df, val_split=0.2, seed=42)

pt_contact_df = pd.read_csv('./data/pt_pdb_train.csv', header=0, index_col=0)
pt_contact_train, pt_contact_val = df_train_test_split(pt_contact_df, val_split=0.2, seed=42)

In [5]:
mp_train_loader = DataLoader(MPDataSet(mp_df=mp_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=True, 
                        immunogenicity=False, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

mp_val_loader = DataLoader(MPDataSet(mp_df=mp_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=True, 
                        immunogenicity=False, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

imm_train_loader = DataLoader(MPDataSet(mp_df=imm_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=True, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

imm_val_loader = DataLoader(MPDataSet(mp_df=imm_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=True, contact=False, mask=None),
                        batch_size= 16, shuffle=True)

mp_contact_train_loader = DataLoader(MPDataSet(mp_df=mp_contact_train, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=False, contact=True, mask=None),
                        batch_size= 2, shuffle=True)

mp_contact_val_loader = DataLoader(MPDataSet(mp_df=mp_contact_val, mhc_type='HLAI', 
                       mhc_max_len=34, pep_max_len=15,
                        binding=False, 
                        immunogenicity=False, contact=True, mask=None),
                        batch_size= 2, shuffle=True)

In [6]:
pt_train_loader = build_loader_for_long_tail(
    dataset=PTDataSet(pt_train, pep_max_len=15, cdr3_max_len=20,
                            binding=True, contact=False, 
                            pep_mask=None, cdr3_mask=0.5),
    peptide_ids = pt_train.peptide, t=1e-3,
    batch_size=16, max_per_group=2,
    alpha=0.5, mix=0.10, repeat_cap=64, seed=42,
    num_workers=0, pin_memory=False
)

pt_val_loader = build_loader_uniform_by_peptide(
    dataset=PTDataSet(pt_val, pep_max_len=15, 
                      cdr3_max_len=20,
                    binding=True, contact=False, 
                    pep_mask=None, cdr3_mask=None),
    peptide_ids = pt_val.peptide, 
    peptides_per_step=64, 
    samples_per_peptide=4, 
    seed=42,
    num_workers=0, pin_memory=False,
    ensure_full_batch=False
)



In [7]:
mpt_train_loader =build_loader_for_long_tail(
    dataset=MPTDataSet(mpt_train, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=True, binding=True, 
               pep_mask=None, cdr3_mask=0.5),
    peptide_ids = mpt_train.peptide,t=1e-3,
    batch_size=16, max_per_group=2,
    alpha=0.5, mix=0.10, repeat_cap=64, seed=42,
    num_workers=0, pin_memory=False
)

mpt_val_loader = build_loader_uniform_by_peptide(
    dataset=MPTDataSet(mpt_val, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=True, binding=True, 
               pep_mask=None, cdr3_mask=None),
    peptide_ids = mpt_val.peptide, 
    peptides_per_step=64, 
    samples_per_peptide=4, 
    seed=42,
    num_workers=0, pin_memory=False,
    ensure_full_batch=False
)

In [8]:
pt_contact_train_loader = DataLoader(PTDataSet(pt_contact_train, pep_max_len=15, cdr3_max_len=20,
                                binding=False, contact=True, 
                                pep_mask=None, cdr3_mask=None),
                                batch_size= 2, shuffle=True)

pt_contact_val_loader = DataLoader(PTDataSet(pt_contact_val, pep_max_len=15, cdr3_max_len=20,
                                binding=False, contact=True, 
                                pep_mask=None, cdr3_mask=None),
                                batch_size= 2, shuffle=True)

In [9]:
align_train_loader = DataLoader(MPTDataSet(align_train, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=False, binding=False, 
               pep_mask=None, cdr3_mask=None),
               batch_size=16, shuffle=True)

align_val_loader = DataLoader(MPTDataSet(align_val, mhc_type='HLAI', 
               mhc_max_len=34, pep_max_len=15, cdr3_max_len=20,
               bv=False, binding=False, 
               pep_mask=None, cdr3_mask=None),
               batch_size=8, shuffle=True)

In [10]:
train_loaders =dict(
    align=align_train_loader, mp=mp_train_loader,pt=pt_train_loader,
    mp_contact=mp_contact_train_loader, pt_contact=pt_contact_train_loader,
    imm=imm_train_loader, mpt=mpt_train_loader
)

val_loaders =dict(
    align=align_val_loader, mp=mp_val_loader,pt=pt_val_loader,
    mp_contact=mp_contact_val_loader, pt_contact=pt_contact_val_loader,
    imm=imm_val_loader, mpt=mpt_val_loader
)

In [11]:
logger = setup_logger()
train_three_phases_multi_loaders(  
    model=model,
    loaders=train_loaders,
    device="mps",
    save_dir="./checkpoints_multi_lora_align_all",
    epochs_A=1, epochs_B=1, epochs_C=1,
    steps_per_epoch=1200, 
    optimizer_ctor=lambda params: torch.optim.AdamW(params, lr=5e-3, weight_decay=0.01),
    grad_accum_steps=1,
    amp=False,
    new_optimizer_each_phase=False,
    log_interval=10,
    task_every = {"mp_contact": 50, "pt_contact": 50},   #
    val_loaders= val_loaders,
    eval_every_epochs=1,
    pep_align=True,
    all_align=True,
    use_lora=True,
    last_n=2,
    cfg_seq_pair=((8,16),(4,8)),
    logger=logger)

[2025-11-14 16:41:06] [Phase A] epoch 1/1 step 10/1200 total=1.5028 | align=0.7804 MP=0.7134 PT=0.6468 contact=0.0000 IMM=0.0000 MPT=0.0000 logic_imm=0.0000 logic_mpt=0.0000
[2025-11-14 16:41:29] [Phase A] epoch 1/1 step 20/1200 total=1.4479 | align=0.5401 MP=0.6968 PT=0.6118 contact=0.0000 IMM=0.0000 MPT=0.0000 logic_imm=0.0000 logic_mpt=0.0000
[2025-11-14 16:41:51] [Phase A] epoch 1/1 step 30/1200 total=1.4216 | align=0.5012 MP=0.6875 PT=0.5966 contact=0.0000 IMM=0.0000 MPT=0.0000 logic_imm=0.0000 logic_mpt=0.0000
[2025-11-14 16:42:13] [Phase A] epoch 1/1 step 40/1200 total=1.4493 | align=0.4889 MP=0.6967 PT=0.6133 contact=0.0000 IMM=0.0000 MPT=0.0000 logic_imm=0.0000 logic_mpt=0.0000


KeyboardInterrupt: 